# Project: IoU and non-max suppression 📦
Every object detector (YOLO, Faster R-CNN, DETR's evaluation) leans on two small algorithms: **intersection over union** to compare boxes, and **non-max suppression** to turn hundreds of overlapping guesses into one box per object. Implement both, check them against `torchvision`, and measure precision before and after NMS on a simulated detector.

Topic: [[Object Detection]], [[Computer Vision Overview]] · guide note: [[Project - IoU and Non-Max Suppression]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches, torch, torchvision
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
rng = np.random.default_rng(0)
objects = np.array([[20, 30, 70, 90], [60, 50, 120, 110], [150, 20, 190, 70]], float)          # ground truth (x1, y1, x2, y2)
def noisy_detector(gt, per_object=15, false_pos=6):
    boxes, scores = [], []
    for b_ in gt:
        for _ in range(per_object): boxes.append(b_ + rng.normal(0, 4, 4)); scores.append(rng.uniform(0.5, 1.0))
    for _ in range(false_pos):
        x, y = rng.uniform(0, 160, 2); w, h = rng.uniform(15, 40, 2); boxes.append([x, y, x + w, y + h]); scores.append(rng.uniform(0.05, 0.45))   # false positives: low confidence
    return np.array(boxes), np.array(scores)
boxes, scores = noisy_detector(objects); print(f"{len(boxes)} raw detections for {len(objects)} objects")

## 1. IoU (one box vs many)
Intersection = overlap rectangle (clip negative widths/heights to 0), union = area A + area B − intersection. Boxes are `(x1, y1, x2, y2)`.

In [ ]:
def iou(box, boxes):
    # TODO 1: IoU of `box` with every row of `boxes`
    raise NotImplementedError  # TODO 1

## 2. Greedy NMS
Sort by score; repeatedly keep the best remaining box and discard every remaining box whose IoU with it is **above** `thresh`. Return the kept indices in order of decreasing score.

In [ ]:
def nms(boxes, scores, thresh=0.5):
    # TODO 2: greedy non-max suppression
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_nms():
    a = np.array([0, 0, 10, 10.0]); bs = np.array([[0, 0, 10, 10], [5, 0, 15, 10], [20, 20, 30, 30.0]])
    ok_iou = np.allclose(iou(a, bs), [1, 50 / 150, 0])
    ref = torchvision.ops.nms(torch.tensor(boxes), torch.tensor(scores), 0.5).tolist()
    return ok_iou and nms(boxes, scores, 0.5) == ref
check("IoU + NMS", _t_nms, "IoU of half-overlapping squares = 50/150; NMS must return exactly torchvision.ops.nms's indices.")
def precision_recall(dets, gt, thr=0.5):
    matched, tp = set(), 0
    for d in dets:
        ious = iou(d, gt); j = int(np.argmax(ious))
        if ious[j] >= thr and j not in matched: matched.add(j); tp += 1
    return tp / max(len(dets), 1), tp / len(gt)
if ready("IoU + NMS"):
    keep = nms(boxes, scores, 0.5); keep_conf = [i for i in keep if scores[i] > 0.5]
    for name, d in [("raw detections", boxes), ("after NMS", boxes[keep]), ("after NMS + score > 0.5", boxes[keep_conf])]:
        p, r = precision_recall(d, objects); print(f"{name:<24} {len(d):>3} boxes  precision {p:.2f}  recall {r:.2f}")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a, d, t in [(ax[0], boxes, "raw"), (ax[1], boxes[keep_conf], "after NMS + threshold")]:
        for b_ in d: a.add_patch(patches.Rectangle(b_[:2], *(b_[2:] - b_[:2]), fill=False, ec="C0", alpha=.6))
        for b_ in objects: a.add_patch(patches.Rectangle(b_[:2], *(b_[2:] - b_[:2]), fill=False, ec="k", lw=2, ls="--"))
        a.set_xlim(0, 210); a.set_ylim(130, 0); a.set_title(t)
    plt.show()
    p, r = precision_recall(boxes[keep_conf], objects)
    check("one box per object", lambda: r == 1.0 and p >= 0.75, "After NMS + a score threshold every object should be found with precision ≥ 0.75.")

<details><summary>▶ Solution</summary>

```python
def iou(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda b_: (b_[..., 2] - b_[..., 0]) * (b_[..., 3] - b_[..., 1])
    return inter / (area(box) + area(boxes) - inter)
```

```python
def nms(boxes, scores, thresh=0.5):
    order = np.argsort(-scores); keep = []
    while len(order):
        i = order[0]; keep.append(int(i))
        if len(order) == 1: break
        rest = order[1:]
        order = rest[iou(boxes[i], boxes[rest]) <= thresh]
    return keep
```
</details>

## Stretch goals
- Soft-NMS: decay scores of overlapping boxes instead of deleting them. Helps for crowded scenes.
- Compute average precision (AP@0.5) by sweeping the score threshold.
- Run a real pretrained detector (`torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn`) on a photo and inspect its raw vs. NMS output.